In [52]:
import numpy as np
import torch
from scipy.linalg import eigh
from scipy.stats import bernoulli
import networkx as nx
import random

# PPNE

In [361]:
def deep_walk(A, walk_length, negative_samples_size, embedding_dim):
    vol_A = torch.sum(A)
    D_inversed = torch.inverse(torch.diag(torch.sum(A, dim=1)))
    sum_matrix = torch.zeros(A.shape)
    for r in range(walk_length):
        sum_matrix += torch.linalg.matrix_power(D_inversed @ A, r+1) @ D_inversed

    Z = torch.log(torch.maximum((vol_A/(walk_length*negative_samples_size)) * sum_matrix, torch.tensor(1)))
    print(Z)
    
    U, S, Vh = torch.linalg.svd(Z + 1e-10 * torch.eye(A.shape[0]), full_matrices=False)

    embeddings = U[:, :embedding_dim] @ torch.sqrt(torch.diag(S)[:embedding_dim, :embedding_dim])
    return embeddings

In [331]:
def privacy_loss(X, pos_edges, neg_edges):
    cosine_similarities = X @ X.T

    pos_similarities = torch.tensor(0.0)
    neg_similarities = torch.tensor(0.0)
    
    for edge in pos_edges:
        pos_similarities += cosine_similarities[edge[0], edge[1]]

    for edge in neg_edges:
        neg_similarities -= cosine_similarities[edge[0], edge[1]]

    return pos_similarities - neg_similarities

In [355]:
def PPNE(A, sampling_size, batching_size, n_iterations, walk_length, negative_samples_size, embedding_dim, pos_edges, neg_edges, k):
    
    A_perturbed = A.copy()
    delta = (A_perturbed == 0) - A_perturbed # change in 1 edge to create candidate perturbations
    current_privacy_gain = 0
    for i in range(n_iterations):

        # Compute Privacy Gains for all pertubations - using torch
        A_perturbed_torch = torch.tensor(A_perturbed, requires_grad=True)
        X = deep_walk(A_perturbed_torch, walk_length, negative_samples_size, embedding_dim)        
        pl = privacy_loss(X, pos_edges, neg_edges)
        pl.backward()
        print(A_perturbed_torch.grad)
        perturbation_privacy_gains = current_privacy_gain - A_perturbed_torch.grad.numpy() * delta
        A_perturbed_torch.grad.zero_()
        
        # Compute Utility Loss for all pertubations - using numpy
        perturbation_utility_losses = proxy_utility_loss(A_perturbed, delta, walk_length, negative_samples_size, embedding_dim)

        # Get the best perturbation
        scores = perturbation_privacy_gains / (perturbation_utility_losses + 1)**k
        scores = scores + np.eye(scores.shape[0]) * -1e10 # exclude the self edges
        idx = np.argmax(scores)
        row = idx // scores.shape[0]
        col = idx % scores.shape[0]

        A_perturbed[row, col] = A_perturbed[row, col] + delta[row, col]
        A_perturbed[col, row] = A_perturbed[col, row] + delta[col, row]
        delta[row, col] = delta[row, col] * -1
        delta[col, row] = delta[row, col] * -1
        current_privacy_gain = perturbation_privacy_gains[row,col]
    
    with torch.no_grad():
        X = deep_walk(torch.tensor(A_perturbed), walk_length, negative_samples_size, embedding_dim)
        
    return X.numpy(), A_perturbed, scores

In [333]:
def proxy_utility_loss(A, delta, walk_length, negative_samples_size, embedding_dim):
    
    # calculating sigma_p
    degrees = np.sum(A, axis=1)
    D = np.diag(degrees)
    generalized_eigh_values, _ = eigh(A, D)
    generalized_eigh_values = generalized_eigh_values[0:A.shape[0]-embedding_dim] # get |V| - K lowest eigenvalues 

    eigenvalue_power_sum = np.zeros(len(generalized_eigh_values))
    for r in range(walk_length):
        eigenvalue_power_sum += np.power(generalized_eigh_values, r+1)
    
    
    d_min = np.min(degrees)
    sigma_p = (1/d_min) * np.abs(eigenvalue_power_sum)

    # calculating the final outputs
    vol_A = np.sum(A)
    output = ((vol_A + 2*delta)/(walk_length*negative_samples_size)) * np.sqrt(np.abs(np.sum(np.power(eigenvalue_power_sum, 2))))
    
    return output

# Processing data

In [344]:
def sample_private_nodes(adj_matrix, private_ratio=0.1):

    # sample a number of private nodes
    num_nodes = adj_matrix.shape[0]

    num_private_nodes = int(num_nodes * private_ratio)

    while True:
        g_copy = nx.Graph(adj_matrix)
        private_nodes = np.random.choice(num_nodes, num_private_nodes, replace=False)
        private_edges = list(g_copy.subgraph(private_nodes).edges())
        # remove private edges
        g_copy.remove_edges_from(private_edges)
        # if isolated nodes exist, remove them
        isolate_nodes = list(nx.isolates(g_copy))

        if len(isolate_nodes) == 0:
            print('no isolated nodes')
            private_edges = np.array(private_edges)
            num_private_edges = private_edges.shape[0]
            private_non_edges = []
            G_private = nx.Graph()
            G_private.add_edges_from(private_edges)
            while len(private_non_edges) < num_private_edges:
                u, v = np.random.choice(private_nodes, 2, replace=False)
                if G_private.has_edge(u, v):
                    continue
                elif [u, v] in private_non_edges:
                    continue
                elif [v, u] in private_non_edges:
                    continue
                else:
                    private_non_edges.append([u, v])
            adj_matrix = nx.to_numpy_array(g_copy) # edited
            private_edges = np.array(private_edges)
            private_non_edges = np.array(private_non_edges)
            break
        else:
            print("isolated nodes exist. re-sample.")
            continue

    return adj_matrix, private_edges, private_non_edges


In [345]:
G = nx.read_edgelist("./datasets/cora/cora.cites", nodetype=int)

In [346]:
np.random.seed(1)
A, pos_edges, neg_edges = sample_private_nodes(nx.to_numpy_array(G))

isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
isolated nodes exist. re-sample.
no isolated nodes


In [362]:
perturbed_embedding, perturbed_adj, s = PPNE(A, sampling_size=10000, batching_size=1, n_iterations=1, 
                                          walk_length=5, negative_samples_size=10, embedding_dim=10, 
                                          pos_edges=pos_edges, neg_edges=neg_edges, k=1)


tensor([[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 2.2313, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 2.8733,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 3.5617, 3.6302, 3.9477],
        [0.0000, 0.0000, 0.0000,  ..., 3.6302, 3.2115, 2.9294],
        [0.0000, 0.0000, 0.0000,  ..., 3.9477, 2.9294, 4.0585]],
       grad_fn=<LogBackward0>)
tensor([[nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        ...,
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan],
        [nan, nan, nan,  ..., nan, nan, nan]], dtype=torch.float64)
tensor([[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 2.2315, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 2.8735,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 3.5619, 3.6304, 

--- TODO: there is some numerical issues with the gradient of svd

In [357]:
perturbed_embedding.shape

(2708, 10)

In [358]:
sum(sum(perturbed_adj- A))

2.0

In [359]:
perturbed_adj

array([[2., 1., 1., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 1., 1.],
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.]])

In [360]:
s

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]])